# Part 2.1 + 2.3 — Naive Bayes and Evaluation
**Owner: Student 2** · Data split: **80% training / 10% validation / 10% testing**, stratified, fixed seed.

Implement multinomial Naive Bayes from scratch using **NumPy** and compare it with scikit-learn's `CountVectorizer` + `MultinomialNB`. Also implement the confusion matrix and per-class/macro precision, recall, F1 from scratch in `utils/evaluation.py`.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

# Support launching Jupyter from the repo root or notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from utils.data_loader import load_emotion_splits
from utils.preprocessing import tokenize, normalize_text
from utils.evaluation import (sklearn_evaluation, confusion_matrix_numpy,
                              metrics_from_confusion_matrix_numpy)

## 2. Shared 80/10/10 split
**Do not resplit in this notebook.** The helper is also used by Student 3. The dataset has `text`, `label`, `label_text` columns; triplet index metadata is intentionally ignored.

In [ ]:
data = load_emotion_splits(seed=SEED)
print('Train/validation/test:', len(data.train), len(data.val), len(data.test))
print('Class ID -> name:', dict(enumerate(data.class_names)))
assert (len(data.train), len(data.val), len(data.test)) == (12800, 1600, 1600)
assert data.num_classes == 6

train_tokens = [tokenize(text) for text in data.train.texts]
val_tokens   = [tokenize(text) for text in data.val.texts]
test_tokens  = [tokenize(text) for text in data.test.texts]

# Identical normalized inputs for custom and sklearn models.
train_texts = [' '.join(tokens) for tokens in train_tokens]
val_texts   = [' '.join(tokens) for tokens in val_tokens]
test_texts  = [' '.join(tokens) for tokens in test_tokens]

## 3. Multinomial Naive Bayes from scratch (NumPy)
**TODO:** Build vocabulary **from training data only**. Compute class priors, token likelihoods with add-one smoothing, and classify using the posterior (use log-space to avoid underflow). Do not use sklearn in the scratch implementation.

In [ ]:
class MultinomialNBScratch:
    def __init__(self, alpha: float = 1.0):
        self.alpha = alpha
        # TODO: add fields (vocabulary, class_log_prior, feature_log_prob, ...)

    def fit(self, tokenized_texts: list[list[str]], y: np.ndarray):
        # TODO: estimate priors and class-conditional likelihoods using NumPy.
        raise NotImplementedError('Student 2: implement Naive Bayes fit')

    def predict(self, tokenized_texts: list[list[str]]) -> np.ndarray:
        # TODO: return class ID with greatest log posterior for each sentence.
        raise NotImplementedError('Student 2: implement Naive Bayes predict')

## 4. Scikit-learn reference (works before scratch model is ready)
Fit the vectorizer on training data only. Tune on validation; reserve test until final evaluation.

In [ ]:
# Baseline. The scratch implementation must NOT call these sklearn classes.
vectorizer = CountVectorizer(token_pattern=r'(?u)\b\w+\b', lowercase=False)
X_train_ref = vectorizer.fit_transform(train_texts)
X_val_ref = vectorizer.transform(val_texts)

sk_nb = MultinomialNB(alpha=1.0)
sk_nb.fit(X_train_ref, data.train.labels)
val_predictions_ref = sk_nb.predict(X_val_ref)
val_metrics_ref = sklearn_evaluation(data.val.labels, val_predictions_ref, data.num_classes)
print('sklearn NB validation macro-F1:', round(val_metrics_ref['macro_f1'], 4))

## 5. Your NumPy model — validation comparison

In [ ]:
# TODO: Uncomment after implementing MultinomialNBScratch.
# my_nb = MultinomialNBScratch(alpha=1.0)
# my_nb.fit(train_tokens, data.train.labels)
# val_predictions_custom = my_nb.predict(val_tokens)
# val_metrics_custom = sklearn_evaluation(data.val.labels,
#                                        val_predictions_custom, data.num_classes)
# print('Custom NB validation macro-F1:', val_metrics_custom['macro_f1'])

## 6. Part 2.3 — Your own confusion matrix and metrics
Implement two shared functions in `utils/evaluation.py`: `confusion_matrix_numpy` and `metrics_from_confusion_matrix_numpy`.

- Confusion matrix shape K×K; rows are true labels and columns predicted labels.
- From TP/FP/FN, calculate precision, recall and F1 for each class, and unweighted macro averages.
- Compare with scikit-learn reference for the **same** predictions, including zero-division handling.

In [ ]:
# TODO: Uncomment when utils/evaluation.py NumPy functions are done.
# cm_custom = confusion_matrix_numpy(data.val.labels, val_predictions_ref, data.num_classes)
# metrics_custom = metrics_from_confusion_matrix_numpy(cm_custom)
# print('Matrices match:', np.array_equal(cm_custom, val_metrics_ref['confusion_matrix']))
# print('Macro F1 matches:', np.isclose(metrics_custom['macro_f1'],
#                                      val_metrics_ref['macro_f1']))

## 7. Final test evaluation and discussion
**TODO:** After selecting your hyperparameters using only train/validation, evaluate custom Naive Bayes and sklearn Naive Bayes on **test**. Present confusion matrices, per-class/macro P/R/F1, and explain any differences. Optionally compare to Student 3 using identical shared splits.